# 05. Reshaping Data: Pivot and Melt

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week09/05.Reshaping-Data-Pivot-and-Melt/notebooks/01_05.Reshaping-Data-Pivot-and-Melt.ipynb)

## Overview
Real-world data often arrives in formats convenient for human entry (wide tables), but analysis, grouping, and visualisation libraries (such as Seaborn and Plotly) typically require **long (tidy) format** where each row represents a single observation and each column represents a variable.

In this notebook, we master:
1. **Wide vs. Long Formats**: Understanding data structures and when to use each.
2. **`pd.melt()`**: Unpivoting wide columns into variable-value pairs.
3. **`DataFrame.pivot()`**: Reshaping long datasets into wide matrices.
4. **`DataFrame.pivot_table()`**: Handling duplicate keys and computing aggregated summaries with row/column totals (`margins=True`).

## 1. Setup: Wide-Format Dataset

Consider student assessment marks across multiple tasks. In wide format, each assessment has its own column.

In [ ]:
import pandas as pd
import numpy as np

# Create sample wide-format student marks
df_wide = pd.DataFrame({
    'student_id': [101, 102, 103, 104, 105],
    'name': ['Liam Nguyen', 'Emma Watson', 'Oliver Brown', 'Sophia Vu', 'Noah Taylor'],
    'campus': ['Sydney', 'Melbourne', 'Sydney', 'Brisbane', 'Melbourne'],
    'assignment_1': [85, 78, 92, 65, 88],
    'assignment_2': [90, 82, 88, 70, 94],
    'final_exam': [88, 75, 95, 72, 91]
})

print("Wide Format Dataset:")
display(df_wide)

## 2. Unpivoting with `pd.melt()` (Wide to Long)

`pd.melt()` collapses multiple columns into key-value pairs.

Key parameters:
- `id_vars`: Columns to keep as identifier variables (not melted).
- `value_vars`: Columns to unpivot into variable-value pairs.
- `var_name`: Name for the new column holding the original column headers.
- `value_name`: Name for the new column holding the values.

In [ ]:
# Reshape wide to long
df_long = pd.melt(
    df_wide,
    id_vars=['student_id', 'name', 'campus'],
    value_vars=['assignment_1', 'assignment_2', 'final_exam'],
    var_name='assessment',
    value_name='score'
)

print("Long (Tidy) Format Dataset:")
display(df_long.head(10))
print(f"\nShape transformed from {df_wide.shape} to {df_long.shape}")

## 3. Reshaping Long to Wide with `DataFrame.pivot()`

When you need to pivot long data back to wide format, use `.pivot()`.

> **Constraint**: `.pivot()` requires each combination of `index` and `columns` to be unique. If there are duplicates, `.pivot()` raises `ValueError: Index contains duplicate entries, cannot reshape`.

In [ ]:
# Pivot long back to wide
df_pivoted = df_long.pivot(
    index=['student_id', 'name', 'campus'],
    columns='assessment',
    values='score'
).reset_index()

# Clear column index name for clean tabular layout
df_pivoted.columns.name = None

print("Restored Wide Format via .pivot():")
display(df_pivoted)

## 4. Handling Duplicates and Summarising with `pivot_table()`

When multiple observations exist for the same index/column pair (e.g. multiple enrolments or repeated measurements), use `DataFrame.pivot_table()` to aggregate values with `aggfunc`.

In [ ]:
# Enrolment data across multiple semesters
df_enrollments = pd.DataFrame({
    'student_id': [101, 101, 102, 102, 103, 103, 104, 104, 105, 105],
    'campus': ['Sydney', 'Sydney', 'Melbourne', 'Melbourne', 'Sydney', 'Sydney', 'Brisbane', 'Brisbane', 'Melbourne', 'Melbourne'],
    'unit': ['ITEC102', 'ITEC105', 'ITEC102', 'ITEC108', 'ITEC102', 'ITEC105', 'ITEC102', 'ITEC108', 'ITEC105', 'ITEC108'],
    'semester': ['Sem 1', 'Sem 2', 'Sem 1', 'Sem 2', 'Sem 1', 'Sem 2', 'Sem 1', 'Sem 2', 'Sem 1', 'Sem 2'],
    'mark': [85, 90, 78, 82, 92, 88, 65, 70, 88, 94]
})

# Pivot table: Mean mark by Campus and Unit
table = pd.pivot_table(
    df_enrollments,
    values='mark',
    index='campus',
    columns='unit',
    aggfunc='mean',
    fill_value=0,
    margins=True,
    margins_name='Campus Average'
)

print("Pivot Table: Mean Marks with Margins (Totals):")
display(table.round(1))

## 5. Practical Exercises

### Exercise 1: Reshape Australian Rainfall from Wide to Long
Below is monthly rainfall (mm) for Australian capital cities.
1. Use `pd.melt()` to reshape the data so each row records `city`, `month`, and `rainfall_mm`.
2. Filter the resulting long DataFrame to only show observations with rainfall greater than 100mm.

In [ ]:
rainfall_wide = pd.DataFrame({
    'city': ['Sydney', 'Melbourne', 'Brisbane', 'Perth'],
    'Jan': [101.3, 44.8, 145.2, 15.4],
    'Feb': [118.0, 48.3, 151.0, 8.8],
    'Mar': [129.7, 52.8, 112.5, 19.7]
})

# --- Student Code Here ---
# Step 1: Melt rainfall_wide
# Step 2: Filter for rainfall_mm > 100

# --- Solution ---
rainfall_long = pd.melt(
    rainfall_wide,
    id_vars=['city'],
    value_vars=['Jan', 'Feb', 'Mar'],
    var_name='month',
    value_name='rainfall_mm'
)
high_rainfall = rainfall_long[rainfall_long['rainfall_mm'] > 100]
display(high_rainfall)

### Exercise 2: Reshaping Long to Wide with Pivot
Take `rainfall_long` from Exercise 1 and pivot it so that `month` is the index and `city` forms the columns.

In [ ]:
# --- Student Code Here ---
# rainfall_pivoted = ...

# --- Solution ---
rainfall_pivoted = rainfall_long.pivot(
    index='month',
    columns='city',
    values='rainfall_mm'
)
display(rainfall_pivoted)

## 6. Key Takeaways

1. **`pd.melt()`**: Converts wide columns to long rows. Essential for preparing datasets for plotting and statistical models.
2. **`DataFrame.pivot()`**: Reshapes long datasets into a wide matrix. Strict requirement: unique index-column pairs.
3. **`DataFrame.pivot_table()`**: Reshapes and aggregates simultaneously when duplicates exist. Supports `aggfunc`, `fill_value`, and `margins=True` for grand totals.